# 06 — LLM-Pipeline 3: Tool-Use (aktive Datenabfrage)

**Idee:** Das LLM erhält nur den Kontext der Aufgabe und eine Beschreibung der verfügbaren Tools. Es entscheidet **selbst**, welche Daten es abruft,
und kann iterativ mehrere Tools aufrufen, bevor es eine Erklärung formuliert.

**Verfügbare Tools:**
| Tool | Beschreibung |
|------|--------------|
| `get_feature_schema` | Metadaten zu allen Features |
| `get_feature_importance` | Globale Feature-Wichtigkeit |
| `get_prediction` | Vorhersage für eigene Feature-Kombination |
| `get_shap_values` | Lokale Beiträge für eine Testinstanz |
| `get_partial_dependence` | Partial-Dependence-Kurve für ein Feature |
| `get_feature_value_context` | Einordnung eines Featurewerts im Trainingsset (Percentile) |
| `get_similar_instances` | k ähnlichste Trainingsinstanzen |
| `get_counterfactual_prediction` | Was-wäre-wenn-Vorhersage bei geänderten Features |


In [1]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib

from utils import INSTANCE_IDS, MODELS_DIR, RESULTS_DIR, PROMPTS_DIR
from utils.data import load_train_test
from utils.tools import ToolBox, TOOL_DEFINITIONS
from utils.llm import DEFAULT_MODEL, _get_client, _with_retry

LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = 2048
OUT_DIR    = RESULTS_DIR / 'pipeline06'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'LLM-Modell:    {MODEL}')
print(f'Testinstanzen: {INSTANCE_IDS}')
print(f'Ausgabe:       {OUT_DIR}')

LLM-Modell:    claude-sonnet-4-6
Testinstanzen: [224, 580, 1041, 1481, 1677, 2058, 2510, 3543, 3847, 4454]
Ausgabe:       /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/pipeline06


## 1 · Daten und Modelle laden

In [2]:
X_train, y_train, X_test, y_test = load_train_test()
from utils.models import load_models
xgb, ebm = load_models(LOSS_KEY)
print(f'X_test: {X_test.shape} | Tools: {[t["name"] for t in TOOL_DEFINITIONS]}')

X_test: (5227, 9) | Tools: ['get_feature_schema', 'get_feature_importance', 'get_prediction', 'get_shap_values', 'get_partial_dependence', 'get_feature_value_context', 'get_similar_instances', 'get_counterfactual_prediction']


## 2 · System-Prompt und Tool-Use-Schleife

In [3]:
SYSTEM_PROMPT = (PROMPTS_DIR / "pipeline_06_tooluse.md").read_text()


def tool_use_loop(
    client,
    toolbox: "ToolBox",
    user_message: str,
    system: str,
    max_rounds: int = 10,
) -> tuple[str, list[dict], int, int, str]:
    """
    Führt die Tool-Use-Schleife aus.
    Gibt zurück: (finaler Text, tool_calls_log, input_tokens, output_tokens, stop_reason)
    stop_reason: 'end_turn' | 'max_tokens' | 'max_rounds' | anderer API-Grund
    """
    messages = [{"role": "user", "content": user_message}]
    total_in, total_out = 0, 0
    last_text = ""
    final_stop_reason = "max_rounds"

    for round_num in range(max_rounds):
        response = _with_retry(
            client.messages.create,
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=[
                {"type": "text", "text": system,
                 "cache_control": {"type": "ephemeral"}}
            ],
            tools=TOOL_DEFINITIONS,
            messages=messages,
        )
        usage = response.usage
        total_in  += usage.input_tokens
        total_out += usage.output_tokens

        messages.append({"role": "assistant", "content": response.content})

        candidate = next(
            (b.text for b in response.content if hasattr(b, "text") and b.text), ""
        )
        if candidate:
            last_text = candidate

        if response.stop_reason == "end_turn":
            return last_text, toolbox.call_log, total_in, total_out, "end_turn"

        if response.stop_reason != "tool_use":
            final_stop_reason = response.stop_reason
            break

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            result = toolbox.dispatch(block.name, block.input)
            tool_results.append({
                "type":        "tool_result",
                "tool_use_id": block.id,
                "content":     json.dumps(result, ensure_ascii=False),
            })
            print(f"    [{round_num+1}] {block.name}({list(block.input.keys())}) -> ok")

        messages.append({"role": "user", "content": tool_results})

    if final_stop_reason == "max_rounds":
        print(f"  [WARN] max_rounds={max_rounds} erreicht – verwende letzten Teiltext.")
    else:
        print(f"  [WARN] stop_reason='{final_stop_reason}' – verwende letzten Teiltext.")
    return last_text or "[Keine Antwort]", toolbox.call_log, total_in, total_out, final_stop_reason


print("Tool-Use-Schleife definiert.")


Tool-Use-Schleife definiert.


## 3 · Aufgaben-Prompt pro Instanz


In [4]:
WEEKDAYS_TASK = {0:"Sonntag", 1:"Montag", 2:"Dienstag", 3:"Mittwoch",
                 4:"Donnerstag", 5:"Freitag", 6:"Samstag"}
MONTHS_TASK   = {1:"Januar", 2:"Februar", 3:"März", 4:"April", 5:"Mai",
                 6:"Juni", 7:"Juli", 8:"August", 9:"September",
                 10:"Oktober", 11:"November", 12:"Dezember"}
WEATHER_TASK  = {1:"klar/wenige Wolken", 2:"Nebel/bewölkt",
                 3:"leichter Regen/Schnee", 4:"Starkregen/Gewitter"}


def build_task_prompt(model_name: str, instance_id: int) -> str:
    row = X_test.iloc[instance_id]
    y   = float(y_test.iloc[instance_id])

    temp_c     = float(row["temp"])   * 41
    hum_pct    = float(row["hum"])    * 100
    wind_kmh   = float(row["windspeed"]) * 67

    lines = [
        f"Ich möchte die Vorhersage für Testinstanz {instance_id} verstehen.",
        f"",
        f"Situation (denormalisierte Werte zur Orientierung):",
        f"  Modell:              {model_name.upper()}",
        f"  Uhrzeit:             {int(row['hr']):02d}:00 Uhr",
        f"  Wochentag:           {WEEKDAYS_TASK.get(int(row['weekday']), int(row['weekday']))}",
        f"  Monat:               {MONTHS_TASK.get(int(row['mnth']), int(row['mnth']))}",
        f"  Jahr:                {'2012' if int(row['yr']) == 1 else '2011'}",
        f"  Wetter:              {WEATHER_TASK.get(int(row['weathersit']), int(row['weathersit']))}",
        f"  Temperatur:          ~{temp_c:.1f} °C",
        f"  Luftfeuchtigkeit:    {hum_pct:.0f} %",
        f"  Windgeschwindigkeit: {wind_kmh:.1f} km/h",
        f"  Feiertag:            {'ja' if int(row['holiday']) == 1 else 'nein'}",
        f"  Tatsächliche Ausleihe: {int(y)} Fahrräder",
        f"",
        f"Bitte nutze mindestens 4 der verfügbaren Tools, um die Vorhersage",
        f"umfassend zu analysieren. Rufe insbesondere get_shap_values und",
        f"get_feature_value_context für die wichtigsten Treiber auf.",
        f"Formuliere anschließend eine verständliche Erklärung auf Deutsch.",
    ]
    return "\n".join(lines)


# Beispiel-Prompt
print(build_task_prompt("xgb", INSTANCE_IDS[0]))


Ich möchte die Vorhersage für Testinstanz 224 verstehen.

Situation (denormalisierte Werte zur Orientierung):
  Modell:              XGB
  Uhrzeit:             19:00 Uhr
  Wochentag:           Donnerstag
  Monat:               Februar
  Jahr:                2011
  Wetter:              klar/wenige Wolken
  Temperatur:          ~8.2 °C
  Luftfeuchtigkeit:    40 %
  Windgeschwindigkeit: 0.0 km/h
  Feiertag:            nein
  Tatsächliche Ausleihe: 96 Fahrräder

Bitte nutze mindestens 4 der verfügbaren Tools, um die Vorhersage
umfassend zu analysieren. Rufe insbesondere get_shap_values und
get_feature_value_context für die wichtigsten Treiber auf.
Formuliere anschließend eine verständliche Erklärung auf Deutsch.


## 4 · LLM-Aufrufe

> **Voraussetzung:** `ANTHROPIC_API_KEY` in `.env` oder als Umgebungsvariable.

In [5]:
client  = _get_client()
results = []
total_in, total_out = 0, 0

for model_name, model_obj in [('xgb', xgb), ('ebm', ebm)]:
    for iid in INSTANCE_IDS:
        toolbox = ToolBox(model_obj, X_train, X_test, y_test, model_name)
        prompt  = build_task_prompt(model_name, iid)
        system  = SYSTEM_PROMPT.replace('{model_name}', model_name.upper())

        print(f'\n{model_name.upper()} inst={iid}')
        t0 = time.time()
        try:
            text, call_log, in_tok, out_tok, stop_reason = tool_use_loop(
                client, toolbox, prompt, system
            )
        except Exception as e:
            print(f'  [ERROR] {type(e).__name__}: {e} – überspringe Instanz.')
            continue
        elapsed = time.time() - t0
        total_in += in_tok; total_out += out_tok

        record = {
            'pipeline':    '06_tooluse',
            'llm_model':   MODEL,
            'loss_key':    LOSS_KEY,
            'xai_model':   model_name,
            'instance_id': iid,
            'explanation': text,
            'stop_reason': stop_reason,
            'tool_calls':  call_log,
            'n_tool_calls': len(call_log),
            'elapsed_s':   round(elapsed, 2),
            'usage':       {'input_tokens': in_tok, 'output_tokens': out_tok},
            'y_true':      float(y_test.iloc[iid]),
        }
        results.append(record)
        (OUT_DIR / f'{model_name}_inst{iid}.json').write_text(
            json.dumps(record, indent=2, ensure_ascii=False)
        )
        print(f'  -> {len(call_log)} Tool-Aufrufe  '
              f'in={in_tok}  out={out_tok}  t={elapsed:.1f}s  stop={stop_reason}')

print(f'\nGesamt:  input={total_in}  output={total_out}')


XGB inst=224
    [1] get_shap_values(['instance_id']) -> ok
    [1] get_feature_importance([]) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_counterfactual_prediction(['instance_id', 'changes']) -> ok
  -> 5 Tool-Aufrufe  in=3075  out=1117  t=34.1s  stop=end_turn

XGB inst=580
    [1] get_shap_values(['instance_id']) -> ok
    [1] get_feature_importance([]) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_counterfactual_prediction(['instance_id', 'changes']) -> ok
  -> 6 Tool-Aufrufe  in=3424  out=1235  t=31.4s  stop=end_turn

XGB inst=1041
    [1] get_shap_values(['instance_id']) -> ok
    [1] get_feature_importance([]) -> ok
    [2] get_feature_value_context(['instance_id', 'feature']) -> ok
    [2] get_fea

## 5 · Beispiel-Erklärung und Tool-Trace

In [6]:
rec = results[0]  # erste XGBoost-Erklärung
sep = '=' * 70
print(sep)
print(f"Modell: {rec['xai_model'].upper()}  |  Instanz: {rec['instance_id']}  "
      f"|  Tool-Aufrufe: {rec['n_tool_calls']}")
print(sep)
print(rec['explanation'])
print()
print('Tool-Trace:')
for i, call in enumerate(rec['tool_calls']):
    print(f"  [{i+1}] {call['tool']}({list(call['arguments'].keys())})")
    print(f"       -> {call['result_preview'][:120]}")

Modell: XGB  |  Instanz: 224  |  Tool-Aufrufe: 5
Alle Daten sind vorhanden – ich verfasse nun die strukturierte Erklärung.

---

Das Modell sagt für diese Donnerstagabend-Stunde im Februar 2011 eine Ausleihe von **112 Fahrrädern** vorher, während tatsächlich **96 Fahrräder** ausgeliehen wurden. Das ist eine leichte Überschätzung von etwa 17 %, was für ein Stundenmodell dieser Art noch im akzeptablen Bereich liegt. Das Modell erfasst die Grunddynamik der Situation also gut, tendiert hier jedoch etwas zu optimistisch.

Die mit Abstand stärkste bremsende Kraft in dieser Stunde ist die **Temperatur von nur ~8,2 °C** – sie drückt die Vorhersage erheblich nach unten. Ein Blick auf die historischen Daten zeigt, warum: Dieser Wert liegt im untersten 5,6. Perzentil aller Trainingsstunden – es handelt sich also um eine ausgesprochen kalte Stunde, die nur bei gut 5 % aller bekannten Fälle vorkommt. Das Modell „weiß", dass bei solcher Kälte deutlich weniger Menschen spontan zum Fahrrad greifen. Gl

## 6 · Zusammenfassung

In [7]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Modell':        r['xai_model'].upper(),
        'Instanz':       r['instance_id'],
        'y_true':        r['y_true'],
        'Tool-Aufrufe':  r['n_tool_calls'],
        'Tools genutzt': ', '.join({c['tool'] for c in r['tool_calls']}),
        'Wörter':        len(r['explanation'].split()),
        'tok_input':     r['usage']['input_tokens'],
        'tok_output':    r['usage']['output_tokens'],
        'Zeit (s)':      r['elapsed_s'],
    }
    for r in results
])
display(summary)

,Modell,Instanz,y_true,Tool-Aufrufe,Tools genutzt,Wörter,tok_input,tok_output,Zeit (s)
0,XGB,224,96.0,5,"get_shap_values, get_feature_importance, get_c...",288,3075,1117,34.14
1,XGB,580,63.0,6,"get_shap_values, get_feature_importance, get_c...",322,3424,1235,31.45
2,XGB,1041,387.0,5,"get_shap_values, get_feature_importance, get_c...",314,3098,1236,31.60
3,XGB,1481,277.0,7,"get_shap_values, get_feature_importance, get_c...",277,4421,1219,33.25
4,XGB,1677,286.0,7,"get_feature_value_context, get_similar_instanc...",301,5472,1388,37.03
5,XGB,2058,243.0,6,"get_shap_values, get_feature_importance, get_c...",301,2891,1237,30.32
6,XGB,2510,372.0,5,"get_shap_values, get_feature_importance, get_c...",357,3094,1312,33.75
7,XGB,3543,286.0,5,"get_shap_values, get_feature_importance, get_c...",327,3094,1235,31.79
8,XGB,3847,531.0,8,"get_feature_value_context, get_partial_depende...",320,5625,1483,36.12
9,XGB,4454,354.0,6,"get_shap_values, get_feature_importance, get_c...",269,3298,1222,31.94
